# Colab TTS Voice Cloning API
### Made by [https://github.com/sharadcodes](https://github.com/sharadcodes)
One-command launchers: each model has its own `start_<model>.sh` that creates an isolated `uv` venv, installs deps, starts a Cloudflare quick tunnel + FastAPI server, and prints the public URL.

**Supported models:** `chatterbox` (Turbo, fast), `vibevoice` (1.5B, expressive), `dia` (1.6B, English).

## Usage
1. Set `MODEL` in the config cell below.
2. Run the launch cell — it clones the repo and runs the launcher.
3. The public `https://*.trycloudflare.com` URL appears in the output.
4. `POST /tts` with `text` + `ref_audio` file → WAV bytes.
5. Or use the OpenAI-compatible `POST /v1/audio/speech` endpoint.

## Files
- `start_chatterbox.sh` / `start_vibevoice.sh` / `start_dia.sh` — per-model launchers
- `start.sh` — thin dispatcher (`MODEL=... ./start.sh`)
- `common.sh` — shared helpers (venv, tunnel, server)
- `server.py` — modular FastAPI server with `TTSBackend` ABC + 3 backends
- `clients/gui.py` — Tkinter GUI client
- `clients/generate_from_file.py` — CLI batch generator from a text file
- `voices/` — drop reference audio clips here for named voice reuse

## 1. Config

In [ ]:
# @title Configuration { run: "auto" }
MODEL = "chatterbox"  # @param ["chatterbox", "vibevoice", "dia"]
PORT = 8000           # @param {type:"integer"}

# Chatterbox variant: "turbo" (fast, 350M), "en" (original English), "mtl" (multilingual V3)
CHATTERBOX_VARIANT = "turbo"  # @param ["turbo", "en", "mtl"]

# VibeVoice DDPM inference steps (lower = faster, lower quality)
VIBEVOICE_STEPS = 20  # @param {type:"integer"}

# Dia generation params
DIA_CFG_SCALE = 4.0   # @param {type:"number"}
DIA_TEMPERATURE = 1.8 # @param {type:"number"}
DIA_TOP_P = 0.90      # @param {type:"number"}

# Repo URL — change if you forked it
REPO_URL = "https://github.com/sharadcodes/colab-tts-api.git"  # @param {type:"string"}

print(f"Model: {MODEL} | Port: {PORT}")

## 2. GPU check

In [ ]:
import torch

if not torch.cuda.is_available():
    raise SystemExit(
        "\n*** NO GPU DETECTED. ***\n"
        "Go to: Runtime menu -> Change runtime type -> Hardware accelerator -> T4 GPU (or better)\n"
        "Then re-run this cell."
    )

gpu_name = torch.cuda.get_device_name(0)
gpu_props = torch.cuda.get_device_properties(0)
vram_gb = gpu_props.total_memory / (1024 ** 3)
major, minor = gpu_props.major, gpu_props.minor

print(f"GPU:          {gpu_name}")
print(f"VRAM:         {vram_gb:.1f} GB")
print(f"Compute cap:  {major}.{minor}")
print(f"CUDA:         {torch.version.cuda}")
print(f"PyTorch:      {torch.__version__}")

warnings = []
if vram_gb < 6:
    warnings.append("VRAM < 6GB: none of the supported models will fit reliably.")
if vram_gb < 10:
    warnings.append("VRAM < 10GB: Dia-1.6B (~10GB) may OOM; use chatterbox or vibevoice.")
if major < 7:
    warnings.append("Compute capability < 7.0: bf16/flash-attention unavailable; slow inference.")
if warnings:
    print("\n*** WARNINGS ***")
    for w in warnings:
        print(f"  - {w}")
else:
    print("\nGPU OK — ready to proceed.")

## 3. Launch

Clones the repo (if needed) and runs `start.sh`, which dispatches to the per-model launcher (`start_chatterbox.sh` / `start_vibevoice.sh` / `start_dia.sh`). The public URL prints below. This cell runs until the server is stopped.

Each model gets its own venv at `venv/<model>/` — switching `MODEL` reuses an already-installed env without reinstalling.

**If you haven't pushed to GitHub yet**, uncomment the upload line instead.

In [ ]:
import os, subprocess, sys

WORK_DIR = "/content/colab-tts-api"

# Option A: clone from GitHub (edit REPO_URL in the config cell)
if not os.path.exists(WORK_DIR):
    subprocess.check_call(["git", "clone", REPO_URL, WORK_DIR])
else:
    # Pull latest if already cloned
    subprocess.run(["git", "-C", WORK_DIR, "pull", "--ff-only"])

# Option B: upload files directly (uncomment if not using GitHub)
# from google.colab import files
# os.makedirs(WORK_DIR, exist_ok=True)
# uploaded = files.upload()
# for fname, content in uploaded.items():
#     with open(os.path.join(WORK_DIR, fname), "wb") as f:
#         f.write(content)

# Build env vars from config
env = os.environ.copy()
env.update({
    "MODEL": MODEL,
    "PORT": str(PORT),
    "CHATTERBOX_VARIANT": CHATTERBOX_VARIANT,
    "VIBEVOICE_STEPS": str(VIBEVOICE_STEPS),
    "DIA_CFG_SCALE": str(DIA_CFG_SCALE),
    "DIA_TEMPERATURE": str(DIA_TEMPERATURE),
    "DIA_TOP_P": str(DIA_TOP_P),
})

# Run the per-model launcher via the dispatcher.
# Each model gets its own venv at venv/<model>/ — switching MODEL reuses
# an already-installed env (swap-and-use) without reinstalling.
os.chdir(WORK_DIR)
subprocess.run(["bash", "start.sh"], env=env)